# M0 pretrained smoke test
Chạy Qwen2.5-VL-3B-Instruct trên 20 mẫu validation cố định. Kết quả được lưu thẳng về Drive; không sử dụng test split để chỉnh prompt.

In [ ]:
import torch
from google.colab import drive

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Hãy chọn Runtime > Change runtime type > GPU"
!nvidia-smi

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path

from google.colab import drive
from PIL import Image

DRIVE_ROOT = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE")
if not Path("/content/drive/MyDrive").is_dir():
    print("Đang kết nối Google Drive...", flush=True)
    drive.mount("/content/drive")
SOURCE = DRIVE_ROOT / "source/project_source.zip"
ARCHIVE = DRIVE_ROOT / "datasets/images/coco_smoke_500.zip"
ANNOTATIONS = DRIVE_ROOT / "datasets/processed/smoke_500.jsonl"
WORK_ROOT = Path("/content/vqa_work/coco")
REPO_ROOT = Path("/content/Vietnamese-VQA-Explanation")
EXPECTED_DATA_SHA256 = "4e0fce747dee9a1beaf36e2617deae6582115020d7c219f315525e329a4024f9"

for required in (SOURCE, ARCHIVE, ANNOTATIONS):
    assert required.is_file(), f"Thiếu file: {required}"
print("[1/4] Đủ source, archive ảnh và annotation trên Drive.", flush=True)
digest = hashlib.sha256()
with ARCHIVE.open("rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_DATA_SHA256, "Archive ảnh sai SHA-256"
print("[2/4] SHA-256 archive ảnh hợp lệ.", flush=True)


def safe_extract(archive, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        assert all(
            (destination / name).resolve().is_relative_to(destination) for name in z.namelist()
        ), "Đường dẫn ZIP không hợp lệ"
        z.extractall(destination)


if not REPO_ROOT.exists():
    safe_extract(SOURCE, Path("/content"))
    print("Đã giải nén source.")
else:
    print("Giữ source đã có; không xóa code trong runtime.")
assert (REPO_ROOT / "pyproject.toml").is_file(), "Source thiếu pyproject.toml"
assert (REPO_ROOT / "scripts/run_m0.py").is_file(), "Source thiếu run_m0.py"
records = [
    json.loads(line)
    for line in ANNOTATIONS.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
paths = sorted({record["image_path"] for record in records})
assert len(paths) == 500, f"Manifest có {len(paths)} ảnh, mong đợi 500"
assert all((WORK_ROOT / p).resolve().is_relative_to(WORK_ROOT.resolve()) for p in paths)
if any(not (WORK_ROOT / p).is_file() for p in paths):
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    safe_extract(ARCHIVE, WORK_ROOT)
print("[3/4] Source và ảnh đã sẵn sàng; đang kiểm tra 500 ảnh...", flush=True)
for relative in paths:
    with Image.open(WORK_ROOT / relative) as image:
        image.verify()
print(f"[4/4] HOÀN TẤT CHUẨN BỊ: {len(paths)} ảnh hợp lệ, {len(records)} câu hỏi.", flush=True)
print("Repo:", REPO_ROOT)
print("Ảnh tạm:", WORK_ROOT)
old_predictions = DRIVE_ROOT / "predictions/M0_smoke.jsonl"
if old_predictions.exists():
    saved = [
        json.loads(line)
        for line in old_predictions.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print(f"M0 trên Drive đã có {len(saved)} dự đoán. Không chạy lại ô M0 để tránh ghi đè.")
print("Cell này chỉ chuẩn bị dữ liệu; chưa train SFT.")

In [ ]:
%cd /content/Vietnamese-VQA-Explanation
%pip install -q -e '.[train]'

In [ ]:
OUTPUT = DRIVE_ROOT / "predictions/M0_smoke.jsonl"
!python scripts/run_m0.py \
  --annotations "{ANNOTATIONS}" \
  --image-root "{WORK_ROOT}" \
  --output "{OUTPUT}" \
  --split validation --limit 20 --seed 20260918 --load-in-4bit

In [ ]:
import json

predictions = [json.loads(line) for line in OUTPUT.read_text(encoding="utf-8").splitlines()]
assert len(predictions) == 20, len(predictions)
valid = sum(item["format_valid"] for item in predictions)
print(f"Hoàn thành M0: {len(predictions)} mẫu; đúng định dạng: {valid}/{len(predictions)}")
for item in predictions[:3]:
    print("\n", item["question"])
    print("GT:", item["reference_answer"])
    print("M0:", item["prediction_answer"])
    print("Giải thích:", item["prediction_explanation"])